### Generating RAG Answers

In [1]:
import pandas as pd

In [2]:
df_ground_truth = pd.read_csv("ground_truth-new_alex.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
ground_truth

[{'question': 'Is it okay to join the course late if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eligible for a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I’m a bit late to the course—what do I need to do to still earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'I registered for the LLM Zoomcamp — when should I expect a confirmation email?',
  'document': '977bf7786c'},
 {'question': 'Do I actually need an acceptance email before I can start the course and hand in homework?',
  'document': '977bf7786c'},
 {'question': 'If I filled out the registration form, does that mean I’m officially on a checked list for the course?',
  'document': '977b

In [4]:
# Load the FAQ documents and the search index

from ingest import load_faq_data, build_index
documents = load_faq_data()
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
# Create a lookup table for the original FAQ documents

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
ground_truth[10]['document']

'489dd1c9d9'

In [7]:
doc_idx[ground_truth[10]['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [8]:
doc_idx[ground_truth[10]['document']]['answer']

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [9]:
## deleting the ground truth data not aligned with json source data

gt_ids = []
for d in ground_truth:
    id = d['document']
    if id not in gt_ids:
        gt_ids.append(id)

# unavailable ids
ua_ids=[]
for id in gt_ids:
    try:
        doc_idx[id]
    except KeyError:
        ua_ids.append(id)

In [10]:
ua_ids

[]

In [11]:
## Running our RAG system

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [12]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [13]:
rec = ground_truth[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'document': '74eb249bbf'}

In [14]:
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course late. If you want a certificate, make sure you submit your project while submissions are still open.'

In [15]:
assistant.total_cost()

0.0004875

In [19]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, make sure you submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [20]:
# create a similar function
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [21]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, make sure you submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [22]:
assistant.reset_usage()

In [23]:
# Import the parallel processing helper from the same utility file

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
assistant.total_cost()

0.0

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/315 [00:00<?, ?it/s]

In [26]:
assistant.total_cost()

0.31579124999999986

In [27]:
answers = []

for answer_record in results:
    answers.append(answer_record)

answers

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes, you can still join the course late. If you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes, you can still join even if you missed the start date. You can start whenever you want, as long as the course materials are available.\n\nIf your goal is to get a certificate, make sure to submit your project while submissions are still being accepted.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still eli

In [28]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)

In [29]:
df_answers

,question,answer_llm,answer_orig,document
0,Is it okay to join the course late if I just f...,"Yes, you can still join the course late. If yo...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,Can I still take this course even if I missed ...,"Yes, you can still join even if you missed the...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,If I join after the course has already started...,Yes — you can still join after the course has ...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Do I need to submit my project before submissi...,"Yes. To get the certificate, you need to submi...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,I’m a bit late to the course—what do I need to...,"To still earn the certificate, you need to:\n\...","Yes, but if you want to receive a certificate,...",74eb249bbf
...,...,...,...,...
310,Why do I get a 401 Client Error when using the...,A 401 Client Error usually means the API key i...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
311,What's the easiest way to force-install reques...,Use this command to install the correct `reque...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
312,Can I install requests straight from the GitHu...,Yes — you can install `requests` directly from...,"If you encounter a 401 Client Error, it may in...",4b30b918bc
313,"If pip keeps pulling requests v2.28, what exac...","Run:\n\n```bash\npip install ""requests @ https...","If you encounter a 401 Client Error, it may in...",4b30b918bc


### LLM as a Judge

In [30]:
df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

# A->Q->A' evaluation

from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [32]:
answers[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'Students don’t use the Zoom link directly. For Office Hours or live/workshop sessions:\n\n- Join via **YouTube Live** on the **DataTalksClub YouTube channel**\n- Submit questions in **Slido** (the link is pinned in chat when live)\n- Check the **announcements channel on Telegram and Slack** for the video URL before the session starts\n\nThe Zoom link itself is only published to instructors/presenters/TAs.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed i

In [33]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [34]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [35]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

In [36]:
# taking one record

rec = answers[0]

prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

Task was destroyed but it is pending!
task: <Task pending name='Task-218' coro=<_async_in_context.<locals>.run_in_context() done, defined at /Users/soumensaha/Downloads/dtc-llm-zoomcamp-2026/Module4/.venv/lib/python3.14/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-219' coro=<Kernel.shell_main() running at /Users/soumensaha/Downloads/dtc-llm-zoomcamp-2026/Module4/.venv/lib/python3.14/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/soumensaha/Downloads/dtc-llm-zoomcamp-2026/Module4/.venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/soumensaha/Downloads/dtc-llm-zoomcamp-2026/Module4/.venv/lib/python3.14/site-packages/pydantic/_internal/_generate_schema.py:997: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  args = tuple(self._resolve_forward_ref(a) if isinstance(a, ForwardRef) else a for a in args)
Task was destroyed but it is pending!
task: <

AnswerEvaluation(reasoning='The AI answer preserves the core meaning of the ground truth: late enrollment is allowed, and certificate eligibility depends on submitting the project before the submission window closes. This is semantically equivalent.', score='good')

In [37]:
calc_price(usage)

{'input_cost': 0.0002205, 'output_cost': 0.0002385, 'total_cost': 0.000459}

In [38]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [39]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. This is semantically equivalent.', score='good')

In [40]:
# judge function

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [41]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/315 [00:00<?, ?it/s]

In [42]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [43]:
calc_total_price(usages)

0.20635649999999978

In [44]:
df_eval = pd.DataFrame(evaluations)

In [45]:
# evaluation analysis
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 293/315 = 93.02%


In [46]:
# bad cases
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,I’m a bit late to the course—what do I need to...,74eb249bbf,bad,The ground truth says the only requirement men...
24,Is peer-review of the capstone project require...,69d122f12e,bad,"The AI answer says peer review is required, wh..."
27,What do I actually need to pass in order to ge...,9f689c185f,bad,The ground truth says the only requirement for...
39,Can you tell me when the course is coming back?,bd31146b0e,bad,The ground truth gives a specific return time:...
63,Which model should I use in chat.completions.c...,152af39a53,bad,The AI answer does not match the ground truth....


In [47]:
doc_idx['bd31146b0e']

{'id': 'bd31146b0e',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'When will the course be offered next?',
 'answer': 'Summer 2027.'}

In [54]:
df_answers[df_answers['document'] == 'bd31146b0e']

,question,answer_llm,answer_orig,document
35,When is the next time this course will be offe...,Summer 2027.,Summer 2027.,bd31146b0e
36,Do you know when the next course session starts?,The next course session starts in **Summer 202...,Summer 2027.,bd31146b0e
37,"Will there be another run of this course, and ...",Yes — the course will be offered next in **Sum...,Summer 2027.,bd31146b0e
38,What's the next scheduled offering for this co...,Summer 2027.,Summer 2027.,bd31146b0e
39,Can you tell me when the course is coming back?,I don't know.,Summer 2027.,bd31146b0e
